In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark.sql("USE CATALOG olist")

In [0]:
sellers_silver = (
    spark.table("olist.bronze.sellers_raw")
    .select(
        F.col("seller_id").cast("string"),
        F.col("seller_zip_code_prefix").cast("int"),
        F.col("seller_city").cast("string"),
        F.col("seller_state").cast("string"),
    )
    .dropDuplicates(["seller_id"])
    .withColumn("_loaded_at", F.current_timestamp())
    .withColumn("_source", F.lit("olist_sellers_dataset.csv"))
)

sellers_silver.write.format("delta").mode("overwrite").saveAsTable("olist.silver.sellers")
print(f"sellers: {sellers_silver.count()} rows")

product_categories (the translation table)

In [0]:
product_categories_silver = (
    spark.table("olist.bronze.category_translation_raw")
    .select(
        F.col("product_category_name").cast("string"),
        F.col("product_category_name_english").alias("category_name_english").cast("string"),
    )
    .dropDuplicates(["product_category_name"])
    .withColumn("_loaded_at", F.current_timestamp())
    .withColumn("_source", F.lit("product_category_name_translation.csv"))
)

product_categories_silver.write.format("delta").mode("overwrite").saveAsTable("olist.silver.product_categories")
print(f"product_categories: {product_categories_silver.count()} rows")

products

In [0]:
products_silver = (
    spark.table("olist.bronze.products_raw")
    .select(
        F.col("product_id").cast("string"),
        F.col("product_category_name").cast("string"),
        F.col("product_name_lenght").alias("product_name_length").cast("int"),
        F.col("product_description_lenght").alias("product_description_length").cast("int"),
        F.col("product_photos_qty").cast("int"),
        F.col("product_weight_g").cast("decimal(10,2)"),
        F.col("product_length_cm").cast("decimal(10,2)"),
        F.col("product_height_cm").cast("decimal(10,2)"),
        F.col("product_width_cm").cast("decimal(10,2)"),
    )
    .dropDuplicates(["product_id"])
    .withColumn("_loaded_at", F.current_timestamp())
    .withColumn("_source", F.lit("olist_products_dataset.csv"))
)

products_silver.write.format("delta").mode("overwrite").saveAsTable("olist.silver.products")
print(f"products: {products_silver.count()} rows")

customers (one row per real person)


In [0]:
customers_silver = (
    spark.table("olist.bronze.customers_raw")
    .select(
        F.col("customer_unique_id").cast("string"),
        F.col("customer_zip_code_prefix").cast("int"),
        F.col("customer_city").cast("string"),
        F.col("customer_state").cast("string"),
    )
    .dropDuplicates(["customer_unique_id"])
    .withColumn("_loaded_at", F.current_timestamp())
    .withColumn("_source", F.lit("olist_customers_dataset.csv"))
)

customers_silver.write.format("delta").mode("overwrite").saveAsTable("olist.silver.customers")
print(f"customers: {customers_silver.count()} rows")

customer_orders (the bridge table)

In [0]:
customer_orders_silver = (
    spark.table("olist.bronze.customers_raw")
    .select(
        F.col("customer_id").cast("string"),
        F.col("customer_unique_id").cast("string"),
    )
    .dropDuplicates(["customer_id"])
    .withColumn("_loaded_at", F.current_timestamp())
    .withColumn("_source", F.lit("olist_customers_dataset.csv"))
)

customer_orders_silver.write.format("delta").mode("overwrite").saveAsTable("olist.silver.customer_orders")
print(f"customer_orders: {customer_orders_silver.count()} rows")

orders


In [0]:
orders_silver = (
    spark.table("olist.bronze.orders_raw")
    .select(
        F.col("order_id").cast("string"),
        F.col("customer_id").cast("string"),
        F.col("order_status").cast("string"),
        F.col("order_purchase_timestamp").cast("timestamp"),
        F.col("order_approved_at").cast("timestamp"),
        F.col("order_delivered_carrier_date").cast("timestamp"),
        F.col("order_delivered_customer_date").cast("timestamp"),
        F.col("order_estimated_delivery_date").cast("timestamp"),
    )
    .dropDuplicates(["order_id"])
    .withColumn("_loaded_at", F.current_timestamp())
    .withColumn("_source", F.lit("olist_orders_dataset.csv"))
)

orders_silver.write.format("delta").mode("overwrite").saveAsTable("olist.silver.orders")
print(f"orders: {orders_silver.count()} rows")

order_items

In [0]:
order_items_silver = (
    spark.table("olist.bronze.order_items_raw")
    .select(
        F.col("order_id").cast("string"),
        F.col("order_item_id").cast("int"),
        F.col("product_id").cast("string"),
        F.col("seller_id").cast("string"),
        F.col("shipping_limit_date").cast("timestamp"),
        F.col("price").cast("decimal(10,2)"),
        F.col("freight_value").cast("decimal(10,2)"),
    )
    .dropDuplicates(["order_id", "order_item_id"])
    .withColumn("_loaded_at", F.current_timestamp())
    .withColumn("_source", F.lit("olist_order_items_dataset.csv"))
)

order_items_silver.write.format("delta").mode("overwrite").saveAsTable("olist.silver.order_items")
print(f"order_items: {order_items_silver.count()} rows")

payments (no dedup needed, but note why)

In [0]:
payments_silver = (
    spark.table("olist.bronze.order_payments_raw")
    .select(
        F.col("order_id").cast("string"),
        F.col("payment_sequential").cast("int"),
        F.col("payment_type").cast("string"),
        F.col("payment_installments").cast("int"),
        F.col("payment_value").cast("decimal(10,2)"),
    )
    .dropDuplicates(["order_id", "payment_sequential"])
    .withColumn("_loaded_at", F.current_timestamp())
    .withColumn("_source", F.lit("olist_order_payments_dataset.csv"))
)

payments_silver.write.format("delta").mode("overwrite").saveAsTable("olist.silver.payments")
print(f"payments: {payments_silver.count()} rows")

reviews (real dedup logic — genuine duplicate order_ids exist here)

In [0]:
review_window = Window.partitionBy("order_id").orderBy(F.col("review_answer_timestamp").desc())

reviews_silver = (
    spark.table("olist.bronze.order_reviews_raw")
    .select(
        F.col("review_id").cast("string"),
        F.col("order_id").cast("string"),
        F.expr("try_cast(review_score as int)").alias("review_score"),
        F.col("review_comment_title").cast("string"),
        F.col("review_comment_message").cast("string"),
        F.expr("try_cast(review_creation_date as timestamp)").alias("review_creation_date"),
        F.expr("try_cast(review_answer_timestamp as timestamp)").alias("review_answer_timestamp"),
    )
    .filter(F.col("order_id").isNotNull())
    .withColumn("rn", F.row_number().over(review_window))
    .filter("rn = 1")
    .drop("rn")
    .withColumn("_loaded_at", F.current_timestamp())
    .withColumn("_source", F.lit("olist_order_reviews_dataset.csv"))
)

reviews_silver.write.format("delta").mode("overwrite").saveAsTable("olist.silver.reviews")
print(f"reviews: {reviews_silver.count()} rows")

malformed_count = reviews_silver.filter(F.col("review_answer_timestamp").isNull()).count()
print(f"rows with malformed/missing review_answer_timestamp: {malformed_count}")

## Data quality note — order_reviews
The raw reviews CSV has two malformed-row patterns caused by unescaped 
commas/newlines inside review_comment_message shifting subsequent columns:

1. 9 rows where corruption reaches the order_id field itself — order_id is 
   NULL. These rows carry no usable order linkage, so they were dropped 
   before load (violates the silver table's NOT NULL constraint on order_id 
   by design).
2. 57 rows where only review_answer_timestamp is malformed. These reviews 
   are otherwise valid (real order_id, real score), so they were kept with 
   review_answer_timestamp set to NULL via try_cast, rather than discarded.

Final row count: 98,689 (from ~99,224 raw), after also deduplicating to one 
review per order_id (keeping the most recent by review_answer_timestamp).